# Verification notebook for Hardy–Littlewood maximal inequality

This notebook is a self-contained mathematical and computational companion to the Hardy–Littlewood maximal inequality section in the paper.

The task template, including the prompt given to the agents, is available [here](https://github.com/dualverse-ai/station/tree/main/example/research_alpha_evolve/hardy_littlewood).

The outline is as follows:

- [1. Problem Definitions](https://nbviewer.org/github/dualverse-ai/station_data_v2/blob/main/artifacts/hardy_littlewood/verification.ipynb#section-1)
- [2. S1. Sharp constants between the centered and uncentered operators.](https://nbviewer.org/github/dualverse-ai/station_data_v2/blob/main/artifacts/hardy_littlewood/verification.ipynb#section-2)
- [3. Verification of the reported 356-atom construction](https://nbviewer.org/github/dualverse-ai/station_data_v2/blob/main/artifacts/hardy_littlewood/verification.ipynb#section-3)

The required packages are:

- `NumPy`


<a id="section-1"></a>

## 1. Problem Definitions

The one-dimensional centered Hardy–Littlewood problem asks how large centered local averages can be relative to the total mass of a function. More precisely, it asks for the least constant $C_0$ such that

$$
|\{M^0f>\lambda\}|\leq \frac{C_0}{\lambda}\|f\|_1,
$$

where

$$
M^0f(x)=\sup_{t>0}\frac1{2t}\int_{x-t}^{x+t}|f(s)|\,ds.
$$

The notebook also uses the non-tangential family

$$
M^\alpha f(x)=
\sup_{\substack{t>0,\ y\in\mathbb R\\ |x-y|\leq\alpha t}}
\frac1{2t}\int_{y-t}^{y+t}|f(s)|\,ds,
\qquad 0\leq\alpha\leq1.
$$

Thus $M^0$ is centered, while $M^1$ is the usual two-sided uncentered operator. Let $C_\alpha$ be the least constant for which

$$
|\{M^\alpha f>\lambda\}|\leq
\frac{C_\alpha}{\lambda}\|f\|_1
\qquad(f\in L^1(\mathbb R),\ \lambda>0).
$$

For a positive finite atomic measure

$$
\mu=\sum_{r=1}^n k_r\delta_{y_r},
\qquad y_1<\cdots<y_n,\quad k_r>0,
$$

write $K_{ij}=\sum_{r=i}^j k_r$.

The next cell implements the interval arithmetic used below.


In [1]:
from fractions import Fraction
from pathlib import Path
import hashlib
import json

import numpy as np

def exact_block_intervals(positions, weights, alpha=Fraction(0)):
    # Return every positive-length block interval and exact gate statistics.
    prefix = [Fraction(0)]
    for weight in weights:
        prefix.append(prefix[-1] + weight)

    intervals = []
    active_by_length = {}
    zero_slack = 0
    inactive = 0
    min_active_slack = None
    min_inactive_slack = None
    reach_factor = 1 + alpha
    for i in range(len(positions)):
        for j in range(i, len(positions)):
            mass = prefix[j + 1] - prefix[i]
            slack = 2 * mass - (positions[j] - positions[i])
            if slack > 0:
                intervals.append(
                    (positions[j] - reach_factor * mass,
                     positions[i] + reach_factor * mass)
                )
                length = j - i + 1
                active_by_length[length] = active_by_length.get(length, 0) + 1
                if min_active_slack is None or slack < min_active_slack:
                    min_active_slack = slack
            elif slack < 0:
                inactive += 1
                miss = -slack
                if min_inactive_slack is None or miss < min_inactive_slack:
                    min_inactive_slack = miss
            else:
                zero_slack += 1
    return intervals, prefix[-1], {
        "inactive": inactive,
        "zero_slack": zero_slack,
        "min_active_slack": min_active_slack,
        "min_inactive_slack": min_inactive_slack,
        "active_by_length": active_by_length,
    }

def merge_intervals(intervals):
    ordered = sorted(intervals)
    components = []
    for left, right in ordered:
        if not components or left > components[-1][1]:
            components.append([left, right])
        elif right > components[-1][1]:
            components[-1][1] = right
    length = sum((right - left for left, right in components), Fraction(0))
    return components, length

def endpoint_sweep_length(intervals):
    events = {}
    for left, right in intervals:
        events[left] = events.get(left, 0) + 1
        events[right] = events.get(right, 0) - 1
    points = sorted(events)
    covered = 0
    length = Fraction(0)
    for index, point in enumerate(points[:-1]):
        covered += events[point]
        next_point = points[index + 1]
        if covered > 0:
            length += next_point - point
    return length


<a id="section-2"></a>

## 2. S1. Sharp constants between the centered and uncentered operators.

### 2.1 Exact finite model

**Lemma 2.1 (exact strict-superlevel block formula).** For every $0\leq\alpha\leq1$,

$$
\{x:M^\alpha\mu(x)>\tfrac12\}
=
\bigcup_{\substack{1\leq i\leq j\leq n\\y_j-y_i<2K_{ij}}}
\bigl(y_j-(1+\alpha)K_{ij},\ y_i+(1+\alpha)K_{ij}\bigr).
\tag{1}
$$

**Proof.** Consider an averaging interval $[L,R]$ of length $d=R-L$. The points allowed by aperture $\alpha$ form

$$
\left[\frac{L+R}{2}-\frac{\alpha d}{2},
      \frac{L+R}{2}+\frac{\alpha d}{2}\right].
$$

Suppose $[L,R]$ contains the block $i,\ldots,j$. Then $L\leq y_i$, $R\geq y_j$, and an average greater than $1/2$ is possible exactly when $d<2K_{ij}$. The feasible triples $(L,R,x)$ form a convex set, so their projection onto the $x$-axis is an interval. Its right endpoint is the supremum of

$$
L+\frac{1+\alpha}{2}d,
$$

namely $y_i+(1+\alpha)K_{ij}$. By symmetry its left endpoint is $y_j-(1+\alpha)K_{ij}$. Such intervals exist exactly when $y_j-y_i<2K_{ij}$.

Every witnessing average captures a consecutive block of atoms and therefore lies in one of these intervals. Conversely, every point in a displayed interval admits an averaging interval of length below $2K_{ij}$ containing that block; any additional atoms only increase the average. This proves (1). $\square$

The strict convention matters. For $M^\alpha\mu\geq1/2$, the activity condition becomes $y_j-y_i\leq2K_{ij}$ and the intervals are closed. If $\alpha>0$, an equality block can then have positive length. All lower bounds below use the strict set in (1).

**Lemma 2.2 (atomic lower bounds give $L^1$ lower bounds).** For every finite positive atomic measure $\mu$,

$$
C_\alpha\geq
\frac{|\{M^\alpha\mu>1/2\}|}{2\mu(\mathbb R)}.
\tag{2}
$$

**Proof.** Replace each atom with a narrow uniform bump,

$$
f_\varepsilon(x)=\sum_i\frac{k_i}{2\varepsilon}
\mathbf 1_{[y_i-\varepsilon,y_i+\varepsilon]}(x).
$$

Then $\|f_\varepsilon\|_1=\mu(\mathbb R)$. Fix $x$ in the atomic strict superlevel set and choose a witnessing interval $[y-t,y+t]$ with $\mu([y-t,y+t])>t$. Enlarging its radius from $t$ to $t+\varepsilon$ captures the full bumps around the atoms in the original interval, preserves the aperture inequality, and still has average greater than $1/2$ for all sufficiently small $\varepsilon$. Hence

$$
\mathbf 1_{\{M^\alpha\mu>1/2\}}(x)
\leq\liminf_{\varepsilon\downarrow0}
\mathbf 1_{\{M^\alpha f_\varepsilon>1/2\}}(x).
$$

Fatou's lemma and the weak inequality for $f_\varepsilon$ give

$$
|\{M^\alpha\mu>1/2\}|
\leq\liminf_{\varepsilon\downarrow0}|\{M^\alpha f_\varepsilon>1/2\}|
\leq2C_\alpha\mu(\mathbb R),
$$

which is (2). $\square$

At $\alpha=0$, (1) becomes the centered interval model

$$
E(\mu)=\bigcup_{i\leq j}(y_j-K_{ij},\ y_i+K_{ij}),
\qquad
R(\mu)=\frac{|E(\mu)|}{2\sum_i k_i}.
\tag{3}
$$

### 2.2 Exact plateau

**Lemma 2.3 (universal upper bound).** For $0\leq\alpha\leq1$, one has $C_\alpha\leq2$.

**Proof.** Every interval allowed for $M^\alpha$ contains $x$, so $M^\alpha f\leq M^1f$. We include the short interval-cover proof that $C_1\leq2$. Fix a compact subset $K\subset\{M^1f>\lambda\}$. A witnessing interval can be enlarged slightly while retaining its strict average, so each covered point may lie in the interior of its witness. Choose a finite subcover of $K$ and successively remove any interval whose removal leaves $K$ covered. In the resulting irredundant family, at most two intervals contain any point: if three shared a point, the intervals with the leftmost left endpoint and rightmost right endpoint would together contain the third. Therefore

$$
\lambda|K|\leq\lambda\sum_j|I_j|
<\sum_j\int_{I_j}|f|
\leq2\|f\|_1.
$$

Exhausting the strict superlevel set by compact subsets proves the result. The same proof applies to a finite positive measure. $\square$

**Proposition 2.4 (equal-chain lower bound).** If $\alpha\geq1/3$, then $C_\alpha\geq2$.

**Proof.** Take $n\geq3$ equal atoms, each of mass $1/n$, with consecutive spacing

$$
g=\frac{4-\eta}{n},\qquad 0<\eta<1.
$$

A block of $\ell$ atoms is active exactly when

$$
(\ell-1)g<\frac{2\ell}{n}.
$$

Thus singletons and adjacent pairs are active, while every $\ell\geq3$ block is inactive because $g>3/n$ and $2\ell/[n(\ell-1)]\leq3/n$.

By Lemma 2.1, a singleton has reach $(1+\alpha)/n$ and an adjacent pair has reach $2(1+\alpha)/n$. The pair interval joins both neighboring singleton intervals when

$$
g<\frac{3(1+\alpha)}n.
$$

For $\alpha\geq1/3$, this follows from $g<4/n\leq3(1+\alpha)/n$. The strict superlevel set is therefore one chain of length

$$
(n-1)g+\frac{2(1+\alpha)}n.
$$

Its total mass is one, so (2) gives

$$
C_\alpha\geq
2-\frac{1-\alpha}{n}-\frac{\eta(n-1)}{2n}.
$$

First let $\eta\downarrow0$ and then $n\to\infty$. This yields $C_\alpha\geq2$. $\square$

**Theorem 2.5 (sharp non-tangential plateau).** For every $1/3\leq\alpha\leq1$,

$$
C_\alpha=2.
$$

**Proof.** Lemma 2.3 gives the upper bound and Proposition 2.4 gives the matching lower bound. $\square$

The values of $C_\alpha$ for $0<\alpha<1/3$, including the smallest onset of the global plateau, remain open.

The next cell checks the equal-chain activity pattern, connected union, and stated score at four apertures and three chain lengths.


In [2]:
def verify_equal_chain(alpha, n, eta):
    alpha = Fraction(alpha)
    eta = Fraction(eta)
    gap = (4 - eta) / n
    positions = [index * gap for index in range(n)]
    weights = [Fraction(1, n)] * n
    chain_intervals, weight, chain_stats = exact_block_intervals(
        positions, weights, alpha
    )
    chain_components, chain_length = merge_intervals(chain_intervals)
    expected_length = (n - 1) * gap + Fraction(2) * (1 + alpha) / n
    expected_score = (
        Fraction(2)
        - (1 - alpha) / n
        - eta * (n - 1) / (2 * n)
    )
    assert set(chain_stats["active_by_length"]) == {1, 2}
    assert len(chain_components) == 1
    assert weight == 1
    assert chain_length == expected_length
    assert chain_length / 2 == expected_score
    return expected_score

rows = []
for alpha in [Fraction(1, 3), Fraction(1, 2), Fraction(3, 4), Fraction(1)]:
    for n in [5, 11, 31]:
        checked_score = verify_equal_chain(alpha, n, Fraction(1, 100))
    rows.append((str(alpha), f"{float(checked_score):.6f}"))

print("Exact equal-chain checks passed for n = 5, 11, 31")
print("alpha, score at n=31 and eta=1/100")
for alpha, value in rows:
    print(f"{alpha:>4}  {value}")


Exact equal-chain checks passed for n = 5, 11, 31
alpha, score at n=31 and eta=1/100
 1/3  1.973656
 1/2  1.979032
 3/4  1.987097
   1  1.995161


### 2.3 Sharp singleton–pair ceiling

The singleton-pair layer has the sharp ceiling $\min\{2,3(1+\alpha)/2\}$, with both branches attained asymptotically by explicit chains. This identifies three-atom blocks as the first mechanism capable of raising the centered score above $3/2$ and narrows the design space for extremal atomic measures across the aperture range.

Let $U_{\mathrm{pair}}^\alpha(\mu)$ be the union in (1) formed only from singleton blocks and active adjacent-pair blocks.

**Theorem 2.6 (sharp pair-layer ceiling).** For every finite positive atomic measure of total mass $W$,

$$
\frac{|U_{\mathrm{pair}}^\alpha(\mu)|}{2W}
\leq
\min\left\{2,\frac{3(1+\alpha)}2\right\}.
\tag{4}
$$

Both branches are sharp as suprema over finite atomic measures.

**Proof.** We first prove the centered estimate $|U_{\mathrm{pair}}^0|\leq3W$. Add atoms from left to right. When the last atom, of mass $b$, is added, translate the preceding atom to $0$, denote its mass by $a$, and denote the new gap by $d>0$. The old union contains $S_0=(-a,a)$. The only new intervals are

$$
S=(d-b,d+b),\qquad
A=(d-a-b,a+b),
$$

with $A$ omitted when inactive. If $A$ is absent, the added length is at most $2b$. If $A$ is present and $d\geq b$, there is no new length left of $S_0$; $A$ adds at most $b$ on the right (and has total length at most $b$ if it misses $S_0$), while $S$ adds at most $2b$. If $d<b$, the left excess is at most $b-d$, while the right excess is at most $b+\max\{0,d-a\}$. Their sum is at most $2b$. Thus each new atom adds at most $3b$. The initial singleton has length $2k_1$, and induction gives $|U_{\mathrm{pair}}^0|\leq3W$.

Set $\kappa=1+\alpha$ and divide all positions by $\kappa$, leaving the weights fixed. After the same spatial scaling, every aperture-$\alpha$ singleton or pair interval is a centered singleton or pair interval for the scaled positions. An active original pair remains active after scaling. Hence the scaled union lies inside a centered pair layer, so

$$
|U_{\mathrm{pair}}^\alpha|\leq3\kappa W.
$$

This gives the second term in (4). The pair layer also lies inside the uncentered level set. Lemma 2.3, applied to the atomic measure, gives $|U_{\mathrm{pair}}^\alpha|\leq4W$, proving the first term.

For sharpness when $\alpha<1/3$, take $n$ equal atoms with gap $3(1+\alpha)/n$; their singleton and pair intervals form a chain up to null contact points, and the score tends to $3(1+\alpha)/2$. When $\alpha\geq1/3$, the equal chains in Proposition 2.4 have pair-layer scores tending to $2$. $\square$

In particular, a centered score above $3/2$ must receive positive-length coverage from a block containing at least three atoms. For $\alpha<1/3$, determining $C_\alpha$ requires the contribution of longer blocks, which can exceed the pair-layer ceiling.



**Station Reference.** The centered pair-layer ceiling comes from Archive #20. The strict block representation and equal-chain proof were developed in Question Room discussion #13 and Archive #50, with the aperture-dependent extension in the Archive #50 addendum. Research Center Evals #774, #778, #781, #783, and #787 checked the formulas, activity gates, strict contacts, endpoint cases, and layer bound.

**Related Work.** [Melas (2002)](https://doi.org/10.1090/S0002-9947-02-02900-8) gives the classical finite-atomic discretization for the centered problem. [Ramos (2019)](https://doi.org/10.5186/aasfm.2019.4409) studied the non-tangential interpolation, proved the sharp variation inequality for $\alpha\geq1/3$, and recorded the weak-type constants between the centered and uncentered endpoints as unknown. The endpoint $C_1=2$ is classical, including for positive measures; see [Bernal (1989)](https://doi.org/10.1017/S030821050001859X) and [Melas (2003)](https://doi.org/10.4007/annals.2003.157.647). The extension $C_\alpha=2$ for $1/3\leq\alpha<1$ and the sharp aperture-dependent pair-layer ceiling appear novel.


<a id="section-3"></a>

## 3. Verification of the reported 356-atom construction

This section verifies the Station construction reported in the paper.


**Theorem 3.1.** There exists a positive atomic measure $\mu$ on $356$ strictly ordered
points whose centered score is

$$
R(\mu)=
\frac{3892672072950587143037}{2500000000000000106902}
>1.55706882918.
$$

Consequently, $C_0>1.55706882918$, and $R(\mu)$ rounds to $1.557069$.

**Verification.** The file `hardy_littlewood_356.npy` contains the positions and weights of this
measure.

The next cell authenticates the stored decimal data, enumerates every consecutive block in (3), merges the resulting rational intervals, and recomputes the union length by an independent endpoint sweep.


In [3]:
DATA_PATH = Path("hardy_littlewood_356.npy")
METADATA_PATH = Path("construction_metadata.json")
EXPECTED_DATA_SHA256 = "6d516f3c5bd4c3ad3c375b3194247ffc537b02986abe9361eeb91436bc37a5d2"
EXPECTED_METADATA_SHA256 = "2efc01c1e134d94744f6241ec6f8a338591edabf548be90fbbed2c233d0a0156"

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

if not DATA_PATH.is_file():
    raise FileNotFoundError(DATA_PATH)
if not METADATA_PATH.is_file():
    raise FileNotFoundError(METADATA_PATH)
assert file_sha256(DATA_PATH) == EXPECTED_DATA_SHA256
assert file_sha256(METADATA_PATH) == EXPECTED_METADATA_SHA256

metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))
assert metadata["artifact_sha256"] == EXPECTED_DATA_SHA256
raw = np.load(DATA_PATH, allow_pickle=False)
assert raw.shape == (2, 356)
assert raw.dtype.kind == "U"
y = [Fraction(value) for value in raw[0].tolist()]
k = [Fraction(value) for value in raw[1].tolist()]
assert all(left < right for left, right in zip(y, y[1:]))
assert all(weight > 0 for weight in k)

print("Authenticated local construction data")
print(f"atoms: {len(y)}; dtype: {raw.dtype}; data SHA-256: {EXPECTED_DATA_SHA256}")

intervals, total_weight, stats = exact_block_intervals(y, k)
components, union_length = merge_intervals(intervals)
sweep_length = endpoint_sweep_length(intervals)
score = union_length / (2 * total_weight)

EXPECTED_WEIGHT = Fraction(1250000000000000053451, 2500000000000000000000)
EXPECTED_UNION = Fraction(3892672072950587143037, 2500000000000000000000)
EXPECTED_SCORE = Fraction(3892672072950587143037, 2500000000000000106902)

assert len(y) * (len(y) + 1) // 2 == 63546
assert len(intervals) == 1913
assert stats["inactive"] == 61633
assert stats["zero_slack"] == 0
assert len(components) == 348
assert stats["min_active_slack"] == Fraction(3, 50000000000000000000)
assert stats["min_inactive_slack"] == Fraction(3, 25000000000000000000)
assert total_weight == EXPECTED_WEIGHT
assert union_length == sweep_length == EXPECTED_UNION
assert score == EXPECTED_SCORE
assert score > Fraction("1.55706882918")

print("Exact exhaustive audit passed")
print(f"blocks examined: 63546; positive-length blocks: {len(intervals)}")
print(f"merged components (up to endpoints): {len(components)}")
print(f"total weight: {total_weight}")
print(f"union length: {union_length}")
print(f"exact score: {score}")
print(f"score rounded to six decimals: {float(score):.6f}")


Authenticated local construction data
atoms: 356; dtype: <U22; data SHA-256: 6d516f3c5bd4c3ad3c375b3194247ffc537b02986abe9361eeb91436bc37a5d2


Exact exhaustive audit passed
blocks examined: 63546; positive-length blocks: 1913
merged components (up to endpoints): 348
total weight: 1250000000000000053451/2500000000000000000000
union length: 3892672072950587143037/2500000000000000000000
exact score: 3892672072950587143037/2500000000000000106902
score rounded to six decimals: 1.557069


**Proof.** The exact rational union length computed above is the displayed fraction, proving the theorem. $\square$

The exact audit finds $1{,}913$ positive-length blocks and no equality blocks. The smallest positive gate margin is $6\times10^{-20}$, while the nearest inactive block misses the gate by $1.2\times10^{-19}$.

**Station Reference.** The construction is recorded in Archive #49 and Research Center Eval #769.

**Related Work.** [Melas (2003)](https://annals.math.princeton.edu/2003/157-2/p08) proved $C_0=(11+\sqrt{61})/12$ and, together with his [2002 construction](https://doi.org/10.1090/S0002-9947-02-02900-8), established finite atomic examples approaching that value. [Georgiev et al. (2025)](https://doi.org/10.48550/arXiv.2511.02864) later reported $1.5080$ in unguided search and approximately $1.533$ with literature guidance. The 356-atom certificate improves those computational benchmarks but not the literature lower bound. The stored configuration and its rational audit appear novel.
